# CHASE-shape epoch82 native test submission on T4

This one cell predicts all 180 original test images with the selected **epoch82 ConvNeXt-Tiny U-Net**. It uses the validation-selected recipe: limb correction, 1024 tiles with 25% overlap, FP16 batch1, threshold0.35, and minimum area300. No YOLO, TTA, public predictions, synthetic-data generation, or training is invoked.

Select a T4 GPU runtime, then run the cell. The original checkpoint and test images stay on Drive. Code and per-image results are also saved there; rerunning this same cell restores the pinned code and reuses verified completed predictions after interruptions.

Only a complete, validated 180-image result is downloaded. Submit the generated CSV to Kaggle; keep the review ZIP for analysis. The notebook does not send a Kaggle submission or calculate test PQ without ground truth.


In [ ]:
# Inference only. Run this same cell again after an interruption or runtime reset.
from pathlib import Path
from google.colab import drive, files
import hashlib, json, os, re, signal, subprocess, sys
BASE = Path("/content/drive/MyDrive/filament-segmentation-2026")
TRAINING_ROOT = BASE / "fixed_split_experiments/chase_shape_best84_recipe_t4_v1"
TEST_IMAGES = BASE / "MAGFiLO_1.0_Kaggle_2026/test/test_images"
OUTPUT = BASE / "submissions/chase_shape_epoch082_native_t4_v1"
CHECKOUT = Path("/content/solar_chase_test_github")
REPOSITORY = "https://github.com/ikjoo5/solar_filament.git"
drive.mount("/content/drive")
OUTPUT = OUTPUT.resolve()
def sha(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()
pointer = OUTPUT / "test_bootstrap_state.json"
if pointer.is_file():
    if pointer.is_symlink():
        raise ValueError("Unsafe saved bootstrap pointer.")
    state = json.loads(pointer.read_text())
    if state.get("scope") != "chase_shape_epoch082_native_t4_v1" or state.get("output_root") != str(OUTPUT):
        raise ValueError("This saved source belongs to another inference run.")
    version = state.get("version", "")
    if not re.fullmatch("[0-9a-f]{16}", version):
        raise ValueError("Invalid saved source version.")
    snapshot = OUTPUT / "code" / version
    manifest_path = snapshot / "source_manifest.json"
    if manifest_path.is_symlink() or not snapshot.resolve().is_relative_to(OUTPUT) or sha(manifest_path) != state["manifest_sha256"]:
        raise ValueError("Saved source manifest changed.")
    manifest = json.loads(manifest_path.read_text())
    canonical = json.dumps(manifest, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()
    if hashlib.sha256(canonical).hexdigest()[:16] != version:
        raise ValueError("Saved source version differs from its manifest.")
    bootstrap = snapshot / "tools/chase_shape_test_colab.py"
    if bootstrap.is_symlink() or not bootstrap.resolve().is_relative_to(snapshot.resolve()):
        raise ValueError("Unsafe saved launcher path.")
    if sha(bootstrap) != manifest.get("tools/chase_shape_test_colab.py"):
        raise ValueError("Saved launcher checksum changed.")
else:
    if CHECKOUT.exists():
        origin = subprocess.check_output(["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True).strip()
        dirty = subprocess.check_output(["git", "-C", str(CHECKOUT), "status", "--porcelain"], text=True).strip()
        if origin != REPOSITORY or dirty:
            raise ValueError("Preserve the existing checkout and choose another CHECKOUT path.")
        subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY, str(CHECKOUT)], check=True)
    bootstrap = CHECKOUT / "tools/chase_shape_test_colab.py"
    if not bootstrap.is_file():
        raise FileNotFoundError("The selected code version lacks the native test launcher.")
command = [sys.executable, "-u", str(bootstrap), "--training-root", str(TRAINING_ROOT),
           "--test-images", str(TEST_IMAGES), "--output", str(OUTPUT)]
# Relay child output through the kernel so Colab displays progress and errors.
process = subprocess.Popen(command, start_new_session=True, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                           errors="replace", bufsize=1)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)
finally:
    if process.poll() is None:
        try: os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError: pass
        try: process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            try: os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError: pass
            process.wait()
    process.stdout.close()
result = json.loads((OUTPUT / "latest_result.json").read_text())
if result.get("status") != "complete" or result.get("images_completed") != 180 or result.get("checkpoint_epoch") != 82:
    raise RuntimeError("A complete epoch82 prediction for all 180 images is required.")
if Path(result.get("output_root", "")).resolve() != OUTPUT:
    raise ValueError("Result belongs to another output directory.")
downloads = []
for key in ("csv", "archive"):
    path = Path(result[key]).resolve()
    if Path(result[key]).is_symlink() or not path.is_relative_to(OUTPUT) or not path.is_file() or sha(path) != result.get(key + "_sha256"):
        raise ValueError("Missing or changed output: " + key)
    downloads.append(path)
for path in downloads:
    print("Download:", path, flush=True)
    files.download(str(path))
